### Conditional Order (Makespan)
For makespan minimisation, suppose two separation-identical aircraft $i$ and $j$ satisfy $r_i \leq r_j$ and $lt_i \leq lt_j$. If the following inequality holds, then placing $i$ before $j$ yields a makespan no worse than the swapped sequence.

$$
W_1 (t_i - b_i)^{\alpha} + W_2 C(t_i, lc_i)
- W_1 (t'_j - b_j)^{\alpha} - W_2 C(t'_j, lc_j)
\;\le\;
W_1 (t'_i - b_i)^{\alpha} + W_2 C(t'_i, lc_i)
- W_1 (t'_i - b_j)^{\alpha} - W_2 C(t'_i, lc_j)
$$

In [1]:
from core.context import *
from core.checks import *

# Retrieve two sequences, only differing by swapping the positions of i and j.
S1, S2, ctx = get_sequences()

# Left and right sides of the new two-aircraft inequality.
def C(t, lc) -> ArithRef: return W2 * If(t <= lc, RealVal(0), If(t <= lc + 300, ω1 * (t - lc) + ω2, ω3 * (t - lc) + ω4))
lhs = W1 * ((S1.takeoff["i"] - ctx.b["i"]) ** α) + W2 * C(S1.takeoff["i"], ctx.lc["i"]) \
    - W1 * ((S2.takeoff["j"] - ctx.b["j"]) ** α) - W2 * C(S2.takeoff["j"], ctx.lc["j"])
rhs = W1 * ((S2.takeoff["i"] - ctx.b["i"]) ** α) + W2 * C(S2.takeoff["i"], ctx.lc["i"]) \
    - W1 * ((S2.takeoff["i"] - ctx.b["j"]) ** α) - W2 * C(S2.takeoff["i"], ctx.lc["j"])

# Encode and enforce the rule premises; total order on release/tw times and that i and j are delta-equivalent.
premises = [
    ctx.r["i"] <= ctx.r["j"],  ctx.lt["i"] <= ctx.lt["j"],
    ctx.b["i"] <= ctx.b["j"],  ctx.lc["i"] <= ctx.lc["j"],
    *ctx.separation_equivalence("i", "j"),
    lhs <= rhs
]

# Encode the claim; no greater makespan in S2 than in S1.
claim = S1.makespan <= S2.makespan

# Check the solver result.
res = verify_pruning_rule(ctx, premises, claim)
print(f"\nConditional Order Makespan (total):           {res}\n")


[14:25:36] Building default sequence pair with ψ-count = 2
[14:25:36] Creating RSP context for 8 aircraft
[14:25:36] Checking non-vacuity with 13 premises and 136 foundational constraints
[14:25:36] Finished solver check for non-vacuity (sat, took 34ms)
[14:25:36] Checking correctness with 13 premises and 136 foundational constraints
[14:25:36] Finished solver check for correctness (unsat, took 80ms)

Conditional Order Makespan (total):           Verified



### Conditional Order (Delay & CTOT)
Using the same release-time, latest-time, and separation-equivalence premises, if the following inequality holds, then the combined delay-and-CTOT dominance check already in this notebook follows.

$$
W_1 (t_i - b_i)^{\alpha} + W_2 C(t_i, lc_i)
- W_1 (t'_j - b_j)^{\alpha} - W_2 C(t'_j, lc_j)
\;\le\;
W_1 (t'_i - b_i)^{\alpha} + W_2 C(t'_i, lc_i)
- W_1 (t'_i - b_j)^{\alpha} - W_2 C(t'_i, lc_j)
$$

In [2]:
from core.context import *
from core.checks import *

# Retrieve two sequences, only differing by swapping the positions of i and j.
S1, S2, ctx = get_sequences()

# Left and right sides of the new two-aircraft inequality.
def C(t, lc) -> ArithRef: return W2 * If(t <= lc, RealVal(0), If(t <= lc + 300, ω1 * (t - lc) + ω2, ω3 * (t - lc) + ω4))
lhs = W1 * ((S1.takeoff["i"] - ctx.b["i"]) ** α) + W2 * C(S1.takeoff["i"], ctx.lc["i"]) \
    - W1 * ((S2.takeoff["j"] - ctx.b["j"]) ** α) - W2 * C(S2.takeoff["j"], ctx.lc["j"])
rhs = W1 * ((S2.takeoff["i"] - ctx.b["i"]) ** α) + W2 * C(S2.takeoff["i"], ctx.lc["i"]) \
    - W1 * ((S2.takeoff["i"] - ctx.b["j"]) ** α) - W2 * C(S2.takeoff["i"], ctx.lc["j"])

# Encode and enforce the rule premises; total order on release/tw times and that i and j are delta-equivalent.
premises = [
    ctx.r["i"] <= ctx.r["j"],  ctx.lt["i"] <= ctx.lt["j"],
    ctx.b["i"] <= ctx.b["j"],  ctx.lc["i"] <= ctx.lc["j"],
    *ctx.separation_equivalence("i", "j"),
    lhs <= rhs
]

# Encode the claim; no aircraft is more delayed in S1 than S2 (aside i and j).
claim = And(*[
    S1.ctot[x] + S1.delay[x] <= S2.ctot[x] + S2.delay[x]
    for x in ctx.aircraft if x not in ("i", "j")
])

# Check the solver result.
res = verify_pruning_rule(ctx, premises, claim)
print(f"\nConditional Order Delay & CTOT (per not i/j ac):  {res}\n")

# Encode the claim; total (delay + ctot) in S1 <= total (delay + ctot) in S2.
claim = (Sum([(S1.ctot[x] + S1.delay[x]) for x in ctx.aircraft])
         <= Sum([(S2.ctot[x] + S2.delay[x]) for x in ctx.aircraft]))

# Check the solver result.
res = verify_pruning_rule(ctx, premises, claim)
print(f"\nConditional Order Delay & CTOT (total):           {res}\n")


[14:25:36] Building default sequence pair with ψ-count = 2
[14:25:36] Creating RSP context for 8 aircraft
[14:25:36] Checking non-vacuity with 13 premises and 136 foundational constraints
[14:25:36] Finished solver check for non-vacuity (sat, took 7ms)
[14:25:36] Checking correctness with 13 premises and 136 foundational constraints
[14:25:36] Finished solver check for correctness (unsat, took 210ms)

Conditional Order Delay & CTOT (per not i/j ac):  Verified

[14:25:36] Checking non-vacuity with 13 premises and 136 foundational constraints
[14:25:36] Finished solver check for non-vacuity (sat, took 5ms)
[14:25:36] Checking correctness with 13 premises and 136 foundational constraints
[14:26:59] Finished solver check for correctness (unknown, took 82627ms)


RuntimeError: Unexpected solver result: unknown

### Conditional Order (Delay)
For delay-based objectives, under $r_i \leq r_j$, $lt_i \leq lt_j$, and separation equivalence, if the following inequality holds, then placing $i$ before $j$ yields a schedule whose delay is no worse than the swapped sequence, both per aircraft and in total.

$$
W_1 (t_i - b_i)^{\alpha} + W_2 C(t_i, lc_i)
- W_1 (t'_j - b_j)^{\alpha} - W_2 C(t'_j, lc_j)
\;\le\;
W_1 (t'_i - b_i)^{\alpha} + W_2 C(t'_i, lc_i)
- W_1 (t'_i - b_j)^{\alpha} - W_2 C(t'_i, lc_j)
$$

In [3]:
from core.context import *
from core.checks import *
from z3 import Sum

# Retrieve two sequences, only differing by swapping the positions of i and j.
S1, S2, ctx = get_sequences()

# Left and right sides of the new two-aircraft inequality.
def C(t, lc) -> ArithRef: return W2 * If(t <= lc, RealVal(0), If(t <= lc + 300, ω1 * (t - lc) + ω2, ω3 * (t - lc) + ω4))
lhs = W1 * ((S1.takeoff["i"] - ctx.b["i"]) ** α) + W2 * C(S1.takeoff["i"], ctx.lc["i"]) \
    - W1 * ((S2.takeoff["j"] - ctx.b["j"]) ** α) - W2 * C(S2.takeoff["j"], ctx.lc["j"])
rhs = W1 * ((S2.takeoff["i"] - ctx.b["i"]) ** α) + W2 * C(S2.takeoff["i"], ctx.lc["i"]) \
    - W1 * ((S2.takeoff["i"] - ctx.b["j"]) ** α) - W2 * C(S2.takeoff["i"], ctx.lc["j"])

# Encode and enforce the rule premises; total order on release/tw times and that i and j are delta-equivalent.
premises = [
    ctx.r["i"] <= ctx.r["j"],  ctx.lt["i"] <= ctx.lt["j"],
    ctx.b["i"] <= ctx.b["j"],  ctx.lc["i"] <= ctx.lc["j"],
    *ctx.separation_equivalence("i", "j"),
    lhs <= rhs
]

# Verify that every aircraft other than i and j incurs no larger delay in S1.
claim = And(*[
    S1.delay[x] <= S2.delay[x]
    for x in ctx.aircraft if x not in ("i", "j")
])

# Ask the solver to certify the claim.
res = verify_pruning_rule(ctx, premises, claim)
print(f"\nConditional Order Delay (per-aircraft): {res}\n")

# Verify that total delay in S1 is no larger than in S2.
claim = Sum([S1.delay[x] for x in ctx.aircraft]) <= Sum([S2.delay[x] for x in ctx.aircraft])

# Ask the solver to certify the claim.
res = verify_pruning_rule(ctx, premises, claim)
print(f"\nConditional Order Delay (total):        {res}\n")


[14:27:02] Building default sequence pair with ψ-count = 2
[14:27:02] Creating RSP context for 8 aircraft
[14:27:03] Checking non-vacuity with 13 premises and 136 foundational constraints
[14:27:03] Finished solver check for non-vacuity (sat, took 11ms)
[14:27:03] Checking correctness with 13 premises and 136 foundational constraints
[14:27:03] Finished solver check for correctness (unsat, took 98ms)

Conditional Order Delay (per-aircraft): Verified

[14:27:03] Checking non-vacuity with 13 premises and 136 foundational constraints
[14:27:03] Finished solver check for non-vacuity (sat, took 6ms)
[14:27:03] Checking correctness with 13 premises and 136 foundational constraints
[14:27:03] Finished solver check for correctness (unsat, took 393ms)

Conditional Order Delay (total):        Verified



### Conditional Order (Time Window Feasibility)
For time-window feasibility, under the same release-time, latest-time, and separation-equivalence premises, if the following inequality holds and the swapped sequence is feasible, then placing $i$ before $j$ does not introduce any new time-window violation.

$$
W_1 (t_i - b_i)^{\alpha} + W_2 C(t_i, lc_i)
- W_1 (t'_j - b_j)^{\alpha} - W_2 C(t'_j, lc_j)
\;\le\;
W_1 (t'_i - b_i)^{\alpha} + W_2 C(t'_i, lc_i)
- W_1 (t'_i - b_j)^{\alpha} - W_2 C(t'_i, lc_j)
$$

In [4]:
from core.context import *
from core.checks import *
from z3 import Implies

# Retrieve two sequences, only differing by swapping the positions of i and j.
S1, S2, ctx = get_sequences()

# Left and right sides of the new two-aircraft inequality.
def C(t, lc) -> ArithRef: return W2 * If(t <= lc, RealVal(0), If(t <= lc + 300, ω1 * (t - lc) + ω2, ω3 * (t - lc) + ω4))
lhs = W1 * ((S1.takeoff["i"] - ctx.b["i"]) ** α) + W2 * C(S1.takeoff["i"], ctx.lc["i"]) \
    - W1 * ((S2.takeoff["j"] - ctx.b["j"]) ** α) - W2 * C(S2.takeoff["j"], ctx.lc["j"])
rhs = W1 * ((S2.takeoff["i"] - ctx.b["i"]) ** α) + W2 * C(S2.takeoff["i"], ctx.lc["i"]) \
    - W1 * ((S2.takeoff["i"] - ctx.b["j"]) ** α) - W2 * C(S2.takeoff["i"], ctx.lc["j"])

# State the premises: order the release and latest times, require separation equivalence, prefer S1 by the new inequality, and assume S2 is feasible.
premises = [
    ctx.r["i"] <= ctx.r["j"],  ctx.lt["i"] <= ctx.lt["j"],
    ctx.b["i"] <= ctx.b["j"],  ctx.lc["i"] <= ctx.lc["j"],
    *ctx.separation_equivalence("i", "j"),
    lhs <= rhs,
    *S2.time_window_feasible
]

# Verify that S1 does not introduce any time-window violation absent from S2.
claim = And(*[
    Implies(S2.window_violation[x], S1.window_violation[x])
    for x in ctx.aircraft
])

# Ask the solver to certify the claim.
res = verify_pruning_rule(ctx, premises, claim)
print(f"\nConditional Order Time Windows: {res}\n")


[14:27:06] Building default sequence pair with ψ-count = 2
[14:27:06] Creating RSP context for 8 aircraft
[14:27:06] Checking non-vacuity with 21 premises and 136 foundational constraints
[14:27:06] Finished solver check for non-vacuity (sat, took 12ms)
[14:27:06] Checking correctness with 21 premises and 136 foundational constraints
[14:27:06] Finished solver check for correctness (unsat, took 14ms)

Conditional Order Time Windows: Verified

